In [ ]:
import time
import json
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
import psycopg2
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()

DB_PARAMS = {
    "dbname" : os.getenv("DB_NAME"),
    "user" : os.getenv("DB_USER"),
    "password" : os.getenv("DB_PASSWORD"),
    "host" : os.getenv("DB_HOST"),
    "port" : os.getenv("DB_PORT")
}

# Path to ChromeDriver
chromedriver_path = "C:\Windows\chromedriver.exe"

# Selenium Setup
chrome_options = Options()
chrome_options.add_argument("--headless")  
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")

In [ ]:
# Start the WebDriver
service = Service(chromedriver_path)
driver = webdriver.Chrome(service=service, options=chrome_options)

In [ ]:
url = "https://www.mcdonalds.com.my/locate-us"
driver.get(url)
time.sleep(2)

In [86]:
state_dropdown = Select(driver.find_element(By.ID, "states"))
state_dropdown.select_by_value("Kuala Lumpur")
time.sleep(2)

In [87]:
def scrape_page():
    # Scrape the page
    outlets = []
    results_div = driver.find_element(By.ID, "results")
    script_tags = results_div.find_elements(By.TAG_NAME, "script")
    
    for script in script_tags:
        script_content = script.get_attribute("innerText").strip()
        try:
            data = json.loads(script_content)
            if isinstance(data, dict) and data.get("@type") == "Restaurant":
                name = data.get("name", "N/A")
                address = data.get("address", "N/A")
                phone = data.get("telephone", "N/A")
                latitude = data["geo"]["latitude"] if "geo" in data else "N/A"
                longitude = data["geo"]["longitude"] if "geo" in data else "N/A"
                menu_url = data.get("menu", "N/A")
                waze_url = data.get("url", "N/A")

                outlets.append({
                    "Name": name,
                    "Address": address,
                    "Phone": phone,
                    "Latitude": latitude,
                    "Longitude": longitude,
                    "Menu": menu_url,
                    "Waze": waze_url
                })
        except json.JSONDecodeError:
            continue
    return outlets


In [88]:
# Scrape the first page
data = scrape_page()

# Handel pagination
while True:
    try:
        next_button = driver.find_element(By.LINK_TEXT, "Next")
        next_button.click()
        time.sleep(3)
        data.extend(scrape_page())
    except:
        break

# Clode the WebDriver
driver.quit()

In [89]:
# Import to PostgreSQL
def save_to_postgresql(data):
    try:
        conn = psycopg2.connect(**DB_PARAMS)
        cursor = conn.cursor()

        # Create table first
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS mcdonalds_outlets (
            id SERIAL PRIMARY KEY,
            Name TEXT,
            Address TEXT,
            Phone TEXT,
            Latitude DOUBLE PRECISION,
            Longitude DOUBLE PRECISION,
            Menu TEXT,
            Waze TEXT
        );
        """)

        # Insert data
        for outlet in data:
            cursor.execute("""
                INSERT INTO mcdonalds_outlets (Name, Address, Phone, Latitude, Longitude, Menu, Waze)
                VALUES (%s, %s, %s, %s, %s, %s, %s);
            """, (outlet["Name"], outlet["Address"], outlet["Phone"], outlet["Latitude"], outlet["Longitude"], outlet["Menu"], outlet["Waze"]))
        
        conn.commit()
        cursor.close()
        conn.close()
        print("Data Imported Successfully!!!!!")
    except Exception as e:
        print(f"Error {e}")

save_to_postgresql(data)

Data Imported Successfully!!!!!


In [ ]:
# Import to Neo4j Database
def save_to_neo4j(data):
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    def add_location(tx, name, phone, address, latitude, longitude, menu_url, waze_url):
        query = """
        MERGE (m:McDonalds {name: $name})
        SET m.address = $address,
        m.phone = $phone,
        m.latitude = $latitude,
        m.longitude = $longitude,
        m.menu_url = $menu_url,
        m.waze_url = $waze_url
        """
        tx.run(query, name = name, phone = phone, address = address, latitude = latitude, longitude = longitude, menu_url = menu_url, waze_url = waze_url)
        with driver.session() as session:
            for outlet in data:
                session.execute_write(add_location, outlet["name"], outlet["phone"], outlet["address"], outlet["latitude"], outlet["longitude"], outlet["menu_url"], outlet["waze_url"])
        
        driver.close()
        print("Data Imported Successfully!!!!!")


save_to_neo4j(data)
print("Data Imported Successfully!!!!!")
